# 1. Inventory Control

- e-commerce
- warehouse
- supply chain

## Problem Concept

ตัดสินใจว่าจะสั่งสินค้าเพิ่มเท่าไร เพื่อให้กำไรสูงสุด โดยคำนึงถึง demand ที่ไม่แน่นอน

- State -> $S_t$ = inventory level
- Action -> $a_t$ = order quantity
- demand -> $d_t$
- inventory transition -> $s_t+1 = max(0, s_t + a_t - d_t)$
- reward (profit) -> $r_t = p \cdot min(s_t + a_t,d_t) - c \cdot a_t - h \dot s_t+1$

| Symbol | Meaning |
|------|------|
| p | selling price |
| c | order cost |
| h | holding cost |

## Bellman Equation (known demand distribution)

$$
V(s) =
\max_a
\mathbb{E}_d
\left[
r(s,a,d) + \gamma V(s')
\right]
$$

### Full Formula

$$
V(s) =
\max_a
\sum_d P(d)
\left[
r(s,a,d) + \gamma V(s')
\right]
$$

In [1]:
import numpy as np

In [8]:
# parameters
max_inventory = 10
max_order = 5

price = 10
order_cost = 4
holding_cost = 1

gamma = 0.9

In [9]:
# demand distribution
demand_probs = {
    0:0.2,
    1:0.3,
    2:0.3,
    3:0.2
}

states = range(max_inventory+1)
actions = range(max_order+1)

V = np.zeros(max_inventory+1)

theta = 1e-4

In [11]:
while True:

    delta = 0

    for s in states:

        v = V[s]

        action_values = []

        for a in actions:

            value = 0

            for d,prob in demand_probs.items():

                sales = min(s + a, d)

                next_state = min(max_inventory, max(0, s + a - d))

                reward = (
                    price*sales
                    - order_cost*a
                    - holding_cost*next_state
                )

                value += prob * (reward + gamma*V[next_state])

            action_values.append(value)

        V[s] = max(action_values)

        delta = max(delta, abs(v-V[s]))

    if delta < theta:
        break

print("Optimal Value Function")
print(V)

Optimal Value Function
[68.99946102 72.99947848 76.99950382 80.99952794 83.29223181 85.0227706
 86.00603951 86.18337535 85.72588859 84.62997178 82.93790107]


# 2. Grid Navigation (Robot Path Planning)

- robotics
- warehouse robot
- autonomous navigation

## Problem Concept

หาเส้นทางที่ดีที่สุดให้ robot เดินไปถึงเป้าหมาย

- state -> $s = (x,y)$
- aCtions -> $A = {up,down,left,right}$
- transition -> $s' = f(s,a)$
- reward -> $
r = 
\begin{cases}
+10 & \text{goal}\\
-1 & \text{step cost}
\end{cases}
$

## Bellman Equation

$$
V(s) =
\max_a
\left[
r + \gamma V(s')
\right]
$$

In [12]:
grid_size = 4
gamma = 0.9

V = np.zeros((grid_size, grid_size))

goal = (3,3)
actions = [
    (0,1),
    (0,-1),
    (1,0),
    (-1,0)
]

theta = 1e-4

In [13]:
while True:
    delta = 0
    
    for x in range(grid_size):
        for y in range(grid_size):
            if (x,y) == goal:
                continue
            
            v = V[x,y]
            values = []
            
            for dx,dy in actions:
                nx = min(grid_size-1,max(0,x+dx))
                ny = min(grid_size-1,max(0,y+dy))
                
                reward = -1
                
                if (nx,ny)==goal:
                    reward = 10
                values.append(
                    reward + gamma*V[nx,ny]
                )
            V[x,y] = max(values)
            delta = max(delta,abs(v-V[x,y]))
    
    if delta < theta:
        break
print(v)

10.0
